# Cell 1: Environment & Keys
(Make sure to put your actual Gemini/OpenAI API key here before running)

In [ ]:
# Cell 1: Initialization
import os
import nest_asyncio
from pydantic_ai import Agent
from pydantic import BaseModel, Field
from typing import Literal

# Required for async execution in Jupyter Notebooks
nest_asyncio.apply()

# WARNING: Do not commit your actual API key to GitHub!
os.environ['GEMINI_API_KEY'] = 'key'

# Cell 2: The Pydantic Shield & Tool Execution
This maps exactly to the Mermaid graph you just generated. Notice how we use Literal to physically block the LLM from hallucinating fake topologies.

In [ ]:
# Cell 2: Strict Schema Definition and Tool Wrapper

class SimulationParams(BaseModel):
    topology: Literal['vortex', 'cdm', 'no_sub'] = Field(
        description="The type of dark matter substructure to simulate."
    )
    main_halo_mass: float = Field(
        description="Mass of the primary lens galaxy in solar masses (e.g., 1e12)."
    )
    sub_mass: float = Field(
        default=0.0, 
        description="Mass of the substructure (vort_mass or cdm mass). Use 0.0 if topology is no_sub."
    )
    inst_name: str = Field(
        description="Telescope instrument specification, e.g., 'Euclid' or 'HST'."
    )

def execute_deeplense_pipeline(params: SimulationParams) -> str:
    """
    Simulates the actual execution of the DeepLense pipeline based on the Mermaid flowchart.
    """
    print("\n" + "="*50)
    print("[HPC SYSTEM] --- TRIGGERING DEEPLENSESIM PIPELINE ---")
    print(f"[HPC SYSTEM] Step 1: Initializing DeepLens(z_halo=0.5, z_gal=1.0)...")
    print(f"[HPC SYSTEM] Step 2: Generating Main Halo (Mass: {params.main_halo_mass:.2e})...")
    
    if params.topology == 'vortex':
        print(f"[HPC SYSTEM] Step 3: Executing make_vortex (Vortex Mass: {params.sub_mass:.2e})...")
    elif params.topology == 'cdm':
        print(f"[HPC SYSTEM] Step 3: Executing make_old_cdm (CDM Mass: {params.sub_mass:.2e})...")
    else:
        print("[HPC SYSTEM] Step 3: Executing make_no_sub (No Substructure)...")
        
    print(f"[HPC SYSTEM] Step 4: Configuring Instrument ({params.inst_name})...")
    print("[HPC SYSTEM] Step 5: Running simple_sim_2 engine...")
    print("[HPC SYSTEM] --- PIPELINE EXECUTION COMPLETE ---")
    print("="*50 + "\n")
    
    return f"SUCCESS: {params.topology} simulation artifact generated using {params.inst_name}."

# Cell 3: The Multi-Agent Orchestrator
This is where we enforce the "Human-in-the-Loop" requirement requested by ML4SCI.

In [ ]:
# Cell 3: Agent Configuration
system_prompt = """
You are the primary orchestration agent for the ML4SCI DeepLenseSim physics pipeline.
Your job is to translate natural language from physicists into strictly validated simulation parameters.

CRITICAL RULES (HUMAN-IN-THE-LOOP):
1. You must collect four parameters: topology ('vortex', 'cdm', or 'no_sub'), main_halo_mass, sub_mass, and inst_name.
2. If the user does not provide all of them, you MUST ask follow-up questions. Do not guess or hallucinate parameters.
3. Once you have all parameters, you MUST print a summary of the configuration and ask the user: "Parameters locked. Do you authorize pipeline execution?"
4. ONLY call the trigger_simulation tool IF the user explicitly replies with an affirmative authorization (e.g., "yes", "authorize", "execute").
5. Do not write Python code for the user. Just use the tool.
"""

# Initialize the Agent
physics_agent = Agent(
    'gemini-1.5-pro', # Updated to a valid Gemini model string
    system_prompt=system_prompt,
)

# Register the Tool
@physics_agent.tool_plain
def trigger_simulation(topology: str, main_halo_mass: float, sub_mass: float, inst_name: str) -> str:
    """Use this tool ONLY when the human user authorizes execution."""
    params = SimulationParams(
        topology=topology,
        main_halo_mass=main_halo_mass, 
        sub_mass=sub_mass, 
        inst_name=inst_name
    )
    return execute_deeplense_pipeline(params)

print("Agentic Orchestrator Online. Ready for Human Interaction.")

# Cell 4: The Execution Loop
This runs the chat interface right inside your notebook output.

In [ ]:
# Cell 4: Interactive Execution Loop
async def run_agentic_workflow():
    print("Welcome to the DeepLenseSim Agentic Orchestrator.")
    print("Type 'exit' to quit.\n")
    
    chat_history = []
    
    while True:
        user_input = input("Researcher: ")
        if user_input.lower() in ['exit', 'quit']:
            break
            
        result = await physics_agent.run(user_input, message_history=chat_history)
        
        print(f"\nAgent: {result.data}\n")
        
        chat_history = result.new_messages()

# Trigger the loop
await run_agentic_workflow()